In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
B = 5
T = 8
d_k = 4
w = 4 
n_heads = 2
Q = torch.randint(0,2,(B,T,d_k),dtype = float)
K = torch.randint(0,2,(B,T,d_k),dtype = float)
V = torch.randint(0,2,(B,T,d_k),dtype = float)
mask_curr = torch.triu(torch.ones(w,w), diagonal=1).bool()
mask_prev= torch.triu(torch.ones(w,w), diagonal=1).bool()

In [ ]:
h = (Q@K.transpose(-2,-1))
h = torch.unsqueeze(h, 0)
h.shape

torch.Size([1, 5, 8, 8])

In [ ]:
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
slopes = torch.tensor([pow(2,-8/n_heads)**i for i in range(1,n_heads+1)],dtype=float)
model = k_pos-q_pos

In [ ]:
Q_chunks = [Q[:,j:j+w,:].float() for j in range(0,T,w)] #Q_chunk[0] : (B,w,d_k)
K_chunks = [K[:,j:j+w,:].float() for j in range(0,T,w)]  #K_chunks.T(-2,-1) : (B,d_k,w) 
# @ = ( B,w,d_k) @ (B,d_k,w) -> (B,w,w) @(B,w,d_k) -> (B,w,d_k)
V_chunks = [V[:,j:j+w,:].float() for j in range(0,T,w)]
chunk_curr = torch.stack([(F.softmax((Q_chunks[i]@K_chunks[i].transpose(-2,-1)/d_k**0.5).masked_fill(mask_curr, float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i]) for i in range(len(Q_chunks))])

chunk_prev = torch.stack([(F.softmax((Q_chunks[i]@K_chunks[i-1].transpose(-2,-1)/d_k**0.5).masked_fill(mask_prev,float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i]) for i in range(1,len(Q_chunks))])
chunk_prev[0]

tensor([[[0.0000, 1.0000, 1.0000, 1.0000],
         [0.6225, 0.3775, 1.0000, 1.0000],
         [0.6667, 0.3333, 0.6667, 0.6667],
         [0.8985, 0.5566, 0.2689, 0.7240]],

        [[1.0000, 1.0000, 0.0000, 1.0000],
         [0.5000, 1.0000, 0.5000, 1.0000],
         [0.7259, 1.0000, 0.5481, 1.0000],
         [0.7500, 0.7500, 0.7500, 1.0000]],

        [[0.0000, 0.0000, 1.0000, 0.0000],
         [0.0000, 0.6225, 1.0000, 0.0000],
         [0.2327, 0.3837, 1.0000, 0.2327],
         [0.1139, 0.6983, 0.8122, 0.3017]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000, 0.5000],
         [0.3333, 0.6667, 0.3333, 0.6667],
         [0.2589, 0.7411, 0.4269, 0.5840]],

        [[1.0000, 0.0000, 1.0000, 0.0000],
         [1.0000, 0.5000, 0.5000, 0.0000],
         [1.0000, 0.5777, 0.4223, 0.4223],
         [1.0000, 0.7240, 0.3775, 0.4551]]])

In [ ]:
chunk_curr = torch.stack([Q_chunks[i]@K_chunks[i].transpose(-2,-1) for i in range(len(Q_chunks))])
(_,B,w,w) = chunk_curr.shape
chunk_curr = ((chunk_curr+(model[:w,:w]*slopes[0]))/d_k**0.5).masked_fill(mask_curr, float('-inf'))
chunk_curr = F.softmax(chunk_curr,dim=-1,dtype = torch.float)
chunk_curr = torch.stack([chunk_curr[i]@V_chunks[i] for i in range(len(chunk_curr))])


chunk_prev = torch.stack([Q_chunks[i]@K_chunks[i-1].transpose(-2,-1) for i in range(1,len(Q_chunks))])
(_,B,w,w) = chunk_prev.shape
chunk_prev = ((chunk_prev+(model[:w,:w]*slopes[0]))/d_k**0.5).masked_fill(mask_prev, float('-inf'))
chunk_prev = F.softmax(chunk_prev,dim=-1,dtype = torch.float)
chunk_prev = torch.stack([chunk_prev[i-1]@V_chunks[i] for i in range(1,len(V_chunks))])
chunk_prev[0]

tensor([[[0.0000, 1.0000, 1.0000, 1.0000],
         [0.6298, 0.3702, 1.0000, 1.0000],
         [0.6770, 0.3230, 0.6562, 0.6562],
         [0.9049, 0.5631, 0.2568, 0.7249]],

        [[1.0000, 1.0000, 0.0000, 1.0000],
         [0.4922, 1.0000, 0.5078, 1.0000],
         [0.7245, 1.0000, 0.5597, 1.0000],
         [0.7540, 0.7382, 0.7616, 1.0000]],

        [[0.0000, 0.0000, 1.0000, 0.0000],
         [0.0000, 0.6298, 1.0000, 0.0000],
         [0.2411, 0.3854, 1.0000, 0.2411],
         [0.1164, 0.7034, 0.8021, 0.3143]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [0.4922, 0.5078, 0.0000, 0.5078],
         [0.3230, 0.6770, 0.3438, 0.6770],
         [0.2471, 0.7529, 0.4337, 0.5883]],

        [[1.0000, 0.0000, 1.0000, 0.0000],
         [1.0000, 0.5078, 0.4922, 0.0000],
         [1.0000, 0.5908, 0.4092, 0.4355],
         [1.0000, 0.7358, 0.3710, 0.4637]]])

In [ ]:
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
k_pos-q_pos
x = chunk_curr



In [ ]:
n_heads = 8
slopes = torch.tensor([pow(2,-8/n_heads)**i for i in range(1,n_heads+1)],dtype=float)

In [ ]:
k = T-1
W = torch.randn((2*k+1,d_k),dtype=float)
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
relative = (q_pos-k_pos).clamp(-k,k)+k #The T*T matrix
#assign an embedding to each i_j : 2k+1 embedding 
relative.shape

torch.Size([8, 8])

In [ ]:
embed = torch.nn.Embedding(2*k+1,d_k)
R = embed(relative)
R.shape

torch.Size([8, 8, 4])

In [ ]:
res = torch.einsum("bid,ijd->bij",Q.float(),R.float())

In [5]:
import torch
self_T = 8
B,T,d_k = 5,3,4 #XT.SHAPE
mask = torch.triu(torch.ones(self_T, self_T), diagonal=1).bool()
mask.shape

torch.Size([8, 8])

In [ ]:
h = torch.randn((B,T,T),dtype=float)
print(h)
h = h.masked_fill(mask[:T,:T], float('-inf'))
print(h,h.shape)


In [ ]:
V = 5
logits = torch.randint(0,2,(1,T,V),dtype=float) #(1,T,V)
req_token_index = torch.argmax(logits[:,-1,:],dim=-1).unsqueeze(0)
logits[:,-1,:].shape

torch.Size([1, 5])